In [1]:
import os
import sys

print("LD_LIBRARY_PATH =", os.environ.get("LD_LIBRARY_PATH", ""))
print("PYTHONPATH =", os.environ.get("PYTHONPATH", ""))

from imu_interfaces.msg import IMUmsg
print(IMUmsg)

LD_LIBRARY_PATH = /home/phillip/EECE5554/LAB3/install/imu_interfaces/lib:/home/phillip/EECE5554/LAB3/install/imu_msgs/lib:/home/phillip/EECE5554/LAB3/install/imu_msgs/lib:/home/phillip/EECE5554/LAB3/install/imu_interfaces/lib:/opt/ros/humble/opt/rviz_ogre_vendor/lib:/opt/ros/humble/lib/x86_64-linux-gnu:/opt/ros/humble/lib
PYTHONPATH = /home/phillip/EECE5554/LAB3/install/imu_interfaces/local/lib/python3.10/dist-packages:/home/phillip/EECE5554/LAB3/install/imu_msgs/local/lib/python3.10/dist-packages:/home/phillip/EECE5554/LAB3/install/imu_msgs/local/lib/python3.10/dist-packages:/home/phillip/EECE5554/LAB3/install/imu_interfaces/local/lib/python3.10/dist-packages:/opt/ros/humble/lib/python3.10/site-packages:/opt/ros/humble/local/lib/python3.10/dist-packages
<class 'imu_interfaces.msg._im_umsg.IMUmsg'>


In [2]:
import rosbag2_py
from rclpy.serialization import deserialize_message
from imu_interfaces.msg import IMUmsg
import numpy as np

bag_path = "/home/phillip/Downloads"

storage_options = rosbag2_py.StorageOptions(
    uri=bag_path,
    storage_id="sqlite3"
)

converter_options = rosbag2_py.ConverterOptions(
    input_serialization_format="cdr",
    output_serialization_format="cdr"
)

reader = rosbag2_py.SequentialReader()
reader.open(storage_options, converter_options)

t = []
gx, gy, gz = [], [], []
ax, ay, az = [], [], []

while reader.has_next():
    topic, data, timestamp = reader.read_next()

    if topic == "/imu":
        msg = deserialize_message(data, IMUmsg)

        t.append(timestamp * 1e-9)

        gx.append(msg.imu.angular_velocity.x)
        gy.append(msg.imu.angular_velocity.y)
        gz.append(msg.imu.angular_velocity.z)

        ax.append(msg.imu.linear_acceleration.x)
        ay.append(msg.imu.linear_acceleration.y)
        az.append(msg.imu.linear_acceleration.z)

t = np.array(t)
gx = np.array(gx, dtype=float)
gy = np.array(gy, dtype=float)
gz = np.array(gz, dtype=float)
ax = np.array(ax, dtype=float)
ay = np.array(ay, dtype=float)
az = np.array(az, dtype=float)

rate = 1 / np.mean(np.diff(t))

print("Samples:", len(t))
print("Sampling rate:", rate)


[INFO] [1773704295.366610454] [rosbag2_storage]: Opened database '/home/phillip/Downloads/5_hour_imu_data_0.db3' for READ_ONLY.


Samples: 914712
Sampling rate: 38.52570618637373


In [3]:
import allantools
import matplotlib.pyplot as plt

def allan_plot(data, title):
    data = data - np.mean(data)

    taus, adev, _, _ = allantools.oadev(
        data,
        rate=rate,
        data_type="freq",
        taus="octave"
    )

    plt.figure(figsize=(7,5))
    plt.loglog(taus, adev, marker="o")
    plt.xlabel("Tau (s)")
    plt.ylabel("Allan deviation")
    plt.title(title)
    plt.grid(True, which="both")
    plt.show()

    return taus, adev

taus_gx, adev_gx = allan_plot(gx, "Gyro X")
taus_gy, adev_gy = allan_plot(gy, "Gyro Y")
taus_gz, adev_gz = allan_plot(gz, "Gyro Z")

taus_ax, adev_ax = allan_plot(ax, "Accel X")
taus_ay, adev_ay = allan_plot(ay, "Accel Y")
taus_az, adev_az = allan_plot(az, "Accel Z")

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 2.2.6
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject